# Partie 5 - Synthese Executive

Format court, redige pour un comite de direction. Ce notebook peut etre exporte en PDF 1 a 2 pages.


In [4]:
from pathlib import Path
import importlib
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import hr_analysis_utils as utils
importlib.reload(utils)

MahalanobisAnomalyDetector = utils.MahalanobisAnomalyDetector
random_oversample_minority = utils.random_oversample_minority
SimpleLogisticRegression = utils.SimpleLogisticRegression
attrition_by_group = utils.attrition_by_group
attrition_gap_summary = utils.attrition_gap_summary
coefficient_importance = utils.coefficient_importance
correlation_with_attrition = utils.correlation_with_attrition
dataset_overview = utils.dataset_overview
kpi_table = utils.kpi_table
load_data = utils.load_data
missing_summary = utils.missing_summary
numeric_summary = utils.numeric_summary
plot_bar = utils.plot_bar
plot_box_by_attrition = utils.plot_box_by_attrition
plot_correlation_heatmap = utils.plot_correlation_heatmap
plot_histogram = utils.plot_histogram
plot_top_coefficients = utils.plot_top_coefficients
prepare_model_data = utils.prepare_model_data
prepare_numeric_anomaly_data = utils.prepare_numeric_anomaly_data
pr_auc_score_manual = utils.pr_auc_score_manual
roc_auc_score_manual = utils.roc_auc_score_manual
salary_gap_by_gender = utils.salary_gap_by_gender
average_by_group = utils.average_by_group
top_attrition_segments = utils.top_attrition_segments
find_best_threshold = utils.find_best_threshold
classification_metrics = utils.classification_metrics

DATA_PATH = PROJECT_ROOT / "raw" / "people_analytics_dataset.csv"
df = load_data(DATA_PATH)
pd.set_option("display.max_columns", 100)


## 1. Ce qu'il faut retenir

- L'entreprise presente un **turnover faible (2,59 %)**, mais avec des poches de risque identifiables.
- Les principaux signaux lies au depart sont un **engagement plus faible**, une **securite psychologique plus basse**, **moins de formation**, et un peu plus d'**absence** et d'**heures supplementaires**.
- Les zones a surveiller en priorite sont la **France**, le **departement IT** et certaines populations managers de niveau 1.


In [5]:
kpi_table(df).head(8)


,KPI,Valeur
0,Effectif total,8020.00
1,Taux d'attrition (%),2.59
2,Score d'engagement moyen,68.15
3,Score de securite psychologique moyen,66.19
4,Taux de promotion sur 3 ans (%),24.64
5,Taux de mobilite interne (%),63.72
6,Heures de formation moyennes,34.64
7,Absenteisme moyen (jours),6.93


## 2. Facteurs cles influencant turnover et engagement

### Turnover

- plus eleve dans certains segments organisationnels
- associe a des conditions d'emploi moins favorables dans ce dataset
- plus difficile a predire a l'echelle individuelle du fait du faible nombre de departs

### Engagement

- plus faible dans les populations qui quittent l'entreprise
- probablement nourri par des leviers de management, de charge de travail, de reconnaissance et de formation


## 3. Resultat du modele predictif

Le modele detecte un **signal faible** :

- il est utile pour orienter la vigilance RH
- il ne doit pas etre utilise seul pour prendre des decisions individuelles
- il confirme l'importance de la formation, de la securite psychologique, de la remuneration et de l'absenteisme dans la lecture du risque
- une experience de **detection d'anomalie** a egalement ete testee : elle remonte un peu plus de departs, mais reste globalement moins robuste que le modele supervise


In [6]:
prepared = prepare_model_data(df, drop_columns=["employee_id"])
model = SimpleLogisticRegression(learning_rate=0.05, epochs=4000, reg_strength=0.02, class_weight="balanced")
model.fit(prepared.X_train, prepared.y_train)
best_threshold = find_best_threshold(prepared.y_val, model.predict_proba(prepared.X_val))
scores = model.predict_proba(prepared.X_test)

pd.DataFrame(
    {
        "Metrique": ["ROC-AUC", "PR-AUC", "Recall", "Precision"],
        "Valeur": [
            roc_auc_score_manual(prepared.y_test, scores),
            pr_auc_score_manual(prepared.y_test, scores),
            classification_metrics(prepared.y_test, scores, threshold=best_threshold["threshold"])["recall"],
            classification_metrics(prepared.y_test, scores, threshold=best_threshold["threshold"])["precision"],
        ],
    }
).round(4)


,Metrique,Valeur
0,ROC-AUC,0.5951
1,PR-AUC,0.0358
2,Recall,0.0476
3,Precision,0.0244


## 4. Recommandations operationnelles

1. Prioriser les plans d'action RH sur les segments les plus exposes au turnover.
2. Integrer l'engagement, la securite psychologique et l'absenteisme dans un dispositif de veille RH trimestriel.
3. Renforcer les parcours de formation et de mobilite interne comme leviers de retention.
4. Lancer une analyse plus fine de l'equite salariale.
5. Ameliorer la qualite des donnees et enrichir le modele avant toute utilisation plus large.
